# Pancake Problem – Cayley Graph Exploration

Systematically explore the **pancake sorting** Cayley graph across different coset groups.

## The Pancake Generators
For a permutation of **n** elements, the generating set consists of all **prefix reversals**:
- **r_i** reverses the first `i` positions (0-indexed), for `i = 2, 3, ..., n`
- This gives **n−1 generators** total (all self-inverse, so the graph is undirected)
- The full graph has **n!** vertices

## Search Space
- **n values**: configurable (e.g. 3 to 12 for full graph, larger for coset graphs)
- **coset types**: 5 groups matching the koltsov3 notebook

## Known diameter facts (for reference)
| n | diameter |
|---|----------|
| 1 | 0 |
| 2 | 1 |
| 3 | 3 |
| 4 | 4 |
| 5 | 5 |
| 6 | 7 |
| 7 | 8 |
| 8 | 9 |
| 9 | 10 |
| 10 | 11 |
| 11 | 13 |
| 12 | 14 |
| 13 | 15 |

## Cell 1: Imports and Setup

In [ ]:
try:
    import numba
    print('Install CayleyPy without dependencies (for Kaggle CPU,GPU):'); print()
    !pip install git+https://github.com/uw-math-ai/cayley-py/tree/parallelized --no-deps -q
except:
    print('Install CayleyPy with dependencies (for Kaggle TPU):'); print()
    !pip install git+https://github.com/uw-math-ai/cayley-py/tree/parallelized -q

# https://github.com/uw-math-ai/cayley-py/tree/parallelized
# https://github.com/cayleypy/cayleypy
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import json
import os
from datetime import datetime
from tqdm.auto import tqdm
from cayleypy import CayleyGraph, PermutationGroups

print("Imports successful!")

Install CayleyPy without dependencies (for Kaggle CPU,GPU):

  error: subprocess-exited-with-error
  
  × git clone --filter=blob:none --quiet https://github.com/uw-math-ai/cayley-py/tree/parallelized /tmp/pip-req-build-a62zyy4q did not run successfully.
  │ exit code: 128
  ╰─> [1 lines of output]
      fatal: repository 'https://github.com/uw-math-ai/cayley-py/tree/parallelized/' not found
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
error: subprocess-exited-with-error

× git clone --filter=blob:none --quiet https://github.com/uw-math-ai/cayley-py/tree/parallelized /tmp/pip-req-build-a62zyy4q did not run successfully.
│ exit code: 128
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
Imports successful!


## Cell 1b: Diagnostics – Inspect CayleyPy API
Run this to confirm which constructor to use for raw generators.

In [4]:
# What methods does PermutationGroups expose?
print("PermutationGroups methods:")
print([m for m in dir(PermutationGroups) if not m.startswith('_')])

# What's importable from cayleypy?
import cayleypy
print("\ncayleypy exports:", [x for x in dir(cayleypy) if not x.startswith('_')])

# Check if CayleyGraphDef exists and what it accepts
try:
    from cayleypy import CayleyGraphDef
    import inspect
    print("\nCayleyGraphDef.__init__ signature:", inspect.signature(CayleyGraphDef.__init__))
except ImportError:
    print("\nCayleyGraphDef not directly importable")

PermutationGroups methods:
['all_cycles', 'all_transpositions', 'block_interchange', 'burnt_pancake', 'conjugacy_classes', 'consecutive_k_cycles', 'coxeter', 'cubic_pancake', 'cyclic_coxeter', 'derangements', 'down_cycles', 'full_reversals', 'generalized_stars', 'increasing_k_cycles', 'involutive_derangements', 'koltsov3', 'larx', 'lrx', 'lsl_cycles', 'lx', 'pancake', 'prefix_cycles', 'rand_generators', 'rapaport_m1', 'rapaport_m2', 'sheveleva2', 'signed_reversals', 'stars', 'three_cycles', 'three_cycles_01i', 'three_cycles_0ij', 'top_spin', 'transposons', 'wrapped_k_cycles']

cayleypy exports: ['BfsResult', 'CayleyGraph', 'CayleyGraphDef', 'CayleyPath', 'GapPuzzles', 'MatrixGenerator', 'MatrixGroups', 'PermutationGroups', 'Predictor', 'Puzzles', 'algo', 'bfs_bitmask', 'bfs_numpy', 'bfs_result', 'cayley_graph', 'cayley_graph_def', 'cayley_path', 'create_graph', 'datasets', 'find_path', 'graphs_lib', 'hasher', 'load_dataset', 'models', 'permutation_utils', 'predictor', 'prepare_graph', 

## Cell 2: Configuration

In [5]:
MIN_N = 3
MAX_N = 12         # For coset graphs. Full graph: keep <= 12 due to n! growth.
OUTPUT_DIR = "results_pancake"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Config: pancake generators, n=[{MIN_N}, {MAX_N}]")
print(f"Output directory: {OUTPUT_DIR}")

Config: pancake generators, n=[3, 12]
Output directory: results_pancake


## Cell 3: Verify Pancake Generator

`PermutationGroups.pancake(n)` is built into cayleypy — no need to define generators manually.
Let's verify it works and inspect what it produces.

In [6]:
# Verify the built-in pancake group definition
for n in [3, 4, 5]:
    defn = PermutationGroups.pancake(n)
    print(f"n={n}: {defn}")

n=3: CayleyGraphDef(generators_type=<GeneratorType.PERMUTATION: 1>, generators_permutations=[[1, 0, 2], [2, 1, 0]], generators_matrices=[], generator_names=['R1', 'R2'], central_state=[0, 1, 2], name='pancake-3')
n=4: CayleyGraphDef(generators_type=<GeneratorType.PERMUTATION: 1>, generators_permutations=[[1, 0, 2, 3], [2, 1, 0, 3], [3, 2, 1, 0]], generators_matrices=[], generator_names=['R1', 'R2', 'R3'], central_state=[0, 1, 2, 3], name='pancake-4')
n=5: CayleyGraphDef(generators_type=<GeneratorType.PERMUTATION: 1>, generators_permutations=[[1, 0, 2, 3, 4], [2, 1, 0, 3, 4], [3, 2, 1, 0, 4], [4, 3, 2, 1, 0]], generators_matrices=[], generator_names=['R1', 'R2', 'R3', 'R4'], central_state=[0, 1, 2, 3, 4], name='pancake-5')


## Cell 4: Coset Group Definitions

Same coset types as the koltsov3 notebook.

In [7]:
COSET_GROUPS = {
    # =========================================================================
    # FULL GRAPH - No coset, explores entire permutation space (n! vertices)
    # =========================================================================
    "full_graph": {
        "FullGraph": lambda n: None,
    },

    # =========================================================================
    # DIFFERENT - First D-1 elements are distinct, rest are all the same
    # Pattern: [0, 1, ..., D-2, D-1, D-1, D-1, ...]
    # =========================================================================
    "different": {
        # n=6: [0, 1, 2, 3, 4, 4]
        "5Different": lambda n: list(range(4)) + [4]*(n-4) if n >= 5 else None,
        # n=7: [0, 1, 2, 3, 4, 5, 5]
        "6Different": lambda n: list(range(5)) + [5]*(n-5) if n >= 6 else None,
        # n=8: [0, 1, 2, 3, 4, 5, 6, 6]
        "7Different": lambda n: list(range(6)) + [6]*(n-6) if n >= 7 else None,
        # n=9: [0, 1, 2, 3, 4, 5, 6, 7, 7]
        "8Different": lambda n: list(range(7)) + [7]*(n-7) if n >= 8 else None,
    },

    # =========================================================================
    # THEN - Blocks of consecutive same values
    # =========================================================================
    "then": {
        # n=6: [0, 0, 0, 1, 1, 1]
        "Binary0then1": lambda n: [0]*(n//2) + [1]*(n - n//2),
        # n=6: [0, 0, 1, 1, 2, 2]
        "0then1then2": lambda n: [0]*(n//3) + [1]*(n//3) + [2]*(n - 2*(n//3)),
        # n=8: [0, 0, 1, 1, 2, 2, 3, 3]
        "0then1then2then3": lambda n: [0]*(n//4) + [1]*(n//4) + [2]*(n//4) + [3]*(n - 3*(n//4)),
        # n=10: [0, 0, 1, 1, 2, 2, 3, 3, 4, 4]
        "0then1then2then3then4": lambda n: [0]*(n//5) + [1]*(n//5) + [2]*(n//5) + [3]*(n//5) + [4]*(n - 4*(n//5)),
    },

    # =========================================================================
    # COINCIDE - Sequential values, last C elements are the same
    # Pattern: [0, 1, 2, ..., n-C-1, n-C, n-C, ..., n-C]
    # =========================================================================
    "coincide": {
        # n=6: [0, 1, 2, 3, 4, 4]
        "2Coincide": lambda n: list(range(n-2)) + [n-2]*2 if n > 2 else None,
        # n=6: [0, 1, 2, 3, 3, 3]
        "3Coincide": lambda n: list(range(n-3)) + [n-3]*3 if n > 3 else None,
        # n=6: [0, 1, 2, 2, 2, 2]
        "4Coincide": lambda n: list(range(n-4)) + [n-4]*4 if n > 4 else None,
        # n=6: [0, 1, 1, 1, 1, 1]
        "5Coincide": lambda n: list(range(n-5)) + [n-5]*5 if n > 5 else None,
        # n=7: [0, 1, 1, 1, 1, 1, 1]
        "6Coincide": lambda n: list(range(n-6)) + [n-6]*6 if n > 6 else None,
    },

    # =========================================================================
    # REPEATS - Repeating pattern of a short block
    # =========================================================================
    "repeats": {
        # n=6: [0, 1, 0, 1, 0, 1]
        "Binary01Repeats": lambda n: [0,1]*(n//2) + [0]*(n - 2*(n//2)),
        # n=6: [0, 1, 0, 1, 0, 1]  (ends with 1 if odd)
        "Binary01Repeats_1": lambda n: [0,1]*(n//2) + [1]*(n - 2*(n//2)),
        # n=6: [0, 1, 2, 0, 1, 2]
        "012Repeats": lambda n: [0,1,2]*(n//3) + [0,1,2][:(n%3)],
        # n=6: [0, 1, 1, 0, 1, 1]
        "011Repeats": lambda n: [0,1,1]*(n//3) + [0,1,1][:(n%3)],
    },
}

print("Coset Groups Summary:")
print("=" * 60)
for group_name, cosets in COSET_GROUPS.items():
    print(f"\n{group_name.upper()}:")
    for coset_name, func in cosets.items():
        example = func(9) if func(9) is not None else "None (full graph)"
        print(f"  {coset_name}: n=9 -> {example}")

Coset Groups Summary:

FULL_GRAPH:
  FullGraph: n=9 -> None (full graph)

DIFFERENT:
  5Different: n=9 -> [0, 1, 2, 3, 4, 4, 4, 4, 4]
  6Different: n=9 -> [0, 1, 2, 3, 4, 5, 5, 5, 5]
  7Different: n=9 -> [0, 1, 2, 3, 4, 5, 6, 6, 6]
  8Different: n=9 -> [0, 1, 2, 3, 4, 5, 6, 7, 7]

THEN:
  Binary0then1: n=9 -> [0, 0, 0, 0, 1, 1, 1, 1, 1]
  0then1then2: n=9 -> [0, 0, 0, 1, 1, 1, 2, 2, 2]
  0then1then2then3: n=9 -> [0, 0, 1, 1, 2, 2, 3, 3, 3]
  0then1then2then3then4: n=9 -> [0, 1, 2, 3, 4, 4, 4, 4, 4]

COINCIDE:
  2Coincide: n=9 -> [0, 1, 2, 3, 4, 5, 6, 7, 7]
  3Coincide: n=9 -> [0, 1, 2, 3, 4, 5, 6, 6, 6]
  4Coincide: n=9 -> [0, 1, 2, 3, 4, 5, 5, 5, 5]
  5Coincide: n=9 -> [0, 1, 2, 3, 4, 4, 4, 4, 4]
  6Coincide: n=9 -> [0, 1, 2, 3, 3, 3, 3, 3, 3]

REPEATS:
  Binary01Repeats: n=9 -> [0, 1, 0, 1, 0, 1, 0, 1, 0]
  Binary01Repeats_1: n=9 -> [0, 1, 0, 1, 0, 1, 0, 1, 1]
  012Repeats: n=9 -> [0, 1, 2, 0, 1, 2, 0, 1, 2]
  011Repeats: n=9 -> [0, 1, 1, 0, 1, 1, 0, 1, 1]


## Cell 5: Helper Functions

In [8]:
def is_valid_central(central):
    """Checks if central has >1 unique value."""
    return central is not None and len(np.unique(central)) > 1


def run_single_experiment(n, coset_name, coset_func):
    """
    Runs BFS for the pancake graph on S_n with the given coset.
    Uses PermutationGroups.pancake(n) which is built into cayleypy.
    Returns BfsResult or None.
    """
    try:
        defn = PermutationGroups.pancake(n)

        central = coset_func(n)
        if coset_name != "FullGraph":
            if not is_valid_central(central):
                return None
            defn = defn.with_central_state(central)

        graph = CayleyGraph(defn)
        return graph.bfs(return_all_edges=False, return_all_hashes=False)

    except Exception as e:
        print(f"Error: {coset_name}, n={n}: {e}")
        return None


def get_group_dir(group_name):
    """Get the output directory for a group, creating it if needed."""
    group_dir = f"{OUTPUT_DIR}/{group_name}"
    os.makedirs(group_dir, exist_ok=True)
    return group_dir


def get_computed_combinations(group_name):
    """Return set of (coset, n) tuples already computed."""
    group_dir = get_group_dir(group_name)
    csv_path = f"{group_dir}/data.csv"
    if not os.path.exists(csv_path):
        return set()
    df = pd.read_csv(csv_path)
    return set(zip(df['coset'], df['n']))


def run_group(group_name, cosets, min_n=MIN_N, max_n=MAX_N,
              coset_filter=None, skip_computed=True):
    """
    Run pancake experiments for a group of cosets, varying n.

    Args:
        group_name:     Name of coset group
        cosets:         Dictionary of coset functions
        min_n:          Minimum n value (default: MIN_N)
        max_n:          Maximum n value (default: MAX_N)
        coset_filter:   Filter which cosets to run:
                          - None: run all
                          - str:  run only that coset
                          - list: run only those cosets
        skip_computed:  If True, skip (coset, n) pairs already in CSV

    Returns:
        Dictionary with results for new computations only
    """
    computed = get_computed_combinations(group_name) if skip_computed else set()

    # Filter cosets
    if coset_filter is None:
        filtered_cosets = cosets
    elif isinstance(coset_filter, str):
        if coset_filter not in cosets:
            raise ValueError(f"Coset '{coset_filter}' not found. Available: {list(cosets.keys())}")
        filtered_cosets = {coset_filter: cosets[coset_filter]}
    else:  # list
        filtered_cosets = {k: v for k, v in cosets.items() if k in coset_filter}
        missing = set(coset_filter) - set(filtered_cosets.keys())
        if missing:
            raise ValueError(f"Cosets not found: {missing}. Available: {list(cosets.keys())}")

    results = {
        "metadata": {
            "graph": "pancake",
            "group": group_name,
            "timestamp": datetime.now().isoformat(),
            "n_range": [min_n, max_n],
            "coset_filter": coset_filter,
        },
        "results": {}
    }

    total_new = 0

    for coset_name, coset_func in filtered_cosets.items():
        skipped = 0
        computed_count = 0
        results["results"][coset_name] = {}

        pbar = tqdm(total=(max_n - min_n + 1),
                    desc=f"{coset_name}",
                    leave=True)

        for n in range(min_n, max_n + 1):
            pbar.set_postfix({"n": n})
            pbar.update(1)

            # Skip if already computed
            if (coset_name, n) in computed:
                skipped += 1
                continue

            result = run_single_experiment(n, coset_name, coset_func)
            if result is not None:
                results["results"][coset_name][f"n={n}"] = {
                    "diameter": result.diameter(),
                    "growth": result.layer_sizes,
                    "last_layer_size": len(result.last_layer())
                }
                computed_count += 1

        pbar.close()
        total_new += computed_count
        print(f"  {coset_name}: Skipped {skipped} cached, computed {computed_count} new")

    print(f"Completed {group_name} ({total_new} new results)")
    return results


def save_results(group_name, results, cosets, append=True):
    """
    Save results to CSV (growth and central stored as JSON strings).

    Args:
        group_name: Name of coset group
        results:    Results dictionary from run_group()
        cosets:     Dictionary of coset functions (to compute central states)
        append:     If True, append to existing CSV; if False, overwrite

    Returns:
        DataFrame with all results (existing + new)
    """
    group_dir = get_group_dir(group_name)
    csv_path = f"{group_dir}/data.csv"

    rows = []
    for coset, n_data in results["results"].items():
        for n_key, metrics in n_data.items():
            n_val = int(n_key.split("=")[1])
            rows.append({
                "coset": coset,
                "n": n_val,
                "diameter": metrics["diameter"],
                "last_layer_size": metrics["last_layer_size"],
                "total_states": sum(metrics["growth"]),
                "growth": json.dumps(metrics["growth"])
            })

    df_new = pd.DataFrame(rows)

    if append and os.path.exists(csv_path):
        df_existing = pd.read_csv(csv_path)
        df = pd.concat([df_existing, df_new], ignore_index=True)
        df = df.drop_duplicates(subset=['coset', 'n'], keep='last')
    else:
        df = df_new

    # Compute central states for all rows
    def compute_central(row):
        coset_func = cosets.get(row['coset'])
        if coset_func is None:
            return None
        central = coset_func(int(row['n']))
        return json.dumps(central) if central is not None else None

    df['central'] = df.apply(compute_central, axis=1)

    df = df.sort_values(['coset', 'n']).reset_index(drop=True)
    df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path} ({len(df)} total rows, {len(df_new)} new)")
    return df


def load_results(group_name):
    """Load results from CSV, parsing growth back to list."""
    group_dir = get_group_dir(group_name)
    df = pd.read_csv(f"{group_dir}/data.csv")
    df['growth'] = df['growth'].apply(json.loads)
    if 'central' in df.columns:
        df['central'] = df['central'].apply(lambda x: json.loads(x) if pd.notna(x) else None)
    return df


def plot_group_results(group_name, df):
    """Create interactive Plotly plots for diameter, growth, and last layer size."""
    group_dir = get_group_dir(group_name)
    coset_names = list(df['coset'].unique())

    df = df.copy()
    df['growth_parsed'] = df['growth'].apply(lambda x: json.loads(x) if isinstance(x, str) else x)

    # ===== PLOT 1: Diameter vs n (lines for each coset) =====
    fig1 = go.Figure()
    for coset_name in coset_names:
        c_df = df[df['coset'] == coset_name].sort_values('n')
        fig1.add_trace(go.Scatter(
            x=c_df['n'], y=c_df['diameter'],
            mode='lines+markers', name=coset_name,
            hovertemplate='n=%{x}<br>diameter=%{y}<br>coset=' + coset_name
        ))

    fig1.update_layout(
        title=dict(text=f'Diameter vs n – Pancake Graph ({group_name})', y=0.95),
        xaxis_title='n', yaxis_title='Diameter',
        legend=dict(x=1.02, y=1), height=600, margin=dict(t=80)
    )
    fig1.write_html(f'{group_dir}/diameter.html')
    fig1.show()

    # ===== PLOT 2: Growth curves (dropdown for coset, lines for n) =====
    fig2 = go.Figure()
    trace_map2 = {}
    trace_idx = 0

    for coset_name in coset_names:
        trace_map2[coset_name] = []
        c_df = df[df['coset'] == coset_name].sort_values('n')
        for _, row in c_df.iterrows():
            growth = row['growth_parsed']
            fig2.add_trace(go.Scatter(
                x=list(range(len(growth))), y=growth,
                mode='lines+markers', name=f"n={row['n']}",
                visible=(coset_name == coset_names[0]),
                hovertemplate='distance=%{x}<br>layer_size=%{y}<br>n=' + str(row['n'])
            ))
            trace_map2[coset_name].append(trace_idx)
            trace_idx += 1

    total_traces2 = trace_idx
    buttons2 = []
    first_button = True
    for coset_name in coset_names:
        if not trace_map2.get(coset_name, []):
            continue
        visibility = [False] * total_traces2
        for idx in trace_map2[coset_name]:
            visibility[idx] = True

        c_df = df[df['coset'] == coset_name]
        growths = c_df['growth_parsed'].tolist()
        if growths:
            max_distance = max(len(g) for g in growths)
            max_layer = max(max(g) for g in growths)
            min_layer = min(min(g) for g in growths if min(g) > 0)
            distance_range = [-0.5, max_distance + 0.5]
            layer_range = [np.log10(min_layer * 0.5), np.log10(max_layer * 2)]
        else:
            distance_range = [0, 10]
            layer_range = [0, 6]

        buttons2.append(dict(label=coset_name, method='update',
                             args=[{'visible': visibility},
                                   {'xaxis.range': distance_range, 'yaxis.range': layer_range}]))
        if first_button:
            init_distance_range = distance_range
            init_layer_range = layer_range
            first_button = False

    fig2.update_layout(
        title=dict(text=f'Growth Curves – Pancake Graph ({group_name})', y=0.95),
        xaxis_title='Distance', yaxis_title='Layer Size', yaxis_type='log',
        xaxis=dict(range=init_distance_range),
        yaxis=dict(range=init_layer_range),
        updatemenus=[dict(buttons=buttons2, direction='down', x=0.0, xanchor='left',
                          y=1.02, yanchor='bottom', showactive=True)],
        legend=dict(x=1.02, y=1), height=650, margin=dict(t=100)
    )
    fig2.write_html(f'{group_dir}/growth.html')
    fig2.show()

    # ===== PLOT 3: Last layer size vs n =====
    fig3 = go.Figure()
    for coset_name in coset_names:
        c_df = df[df['coset'] == coset_name].sort_values('n')
        fig3.add_trace(go.Scatter(
            x=c_df['n'], y=c_df['last_layer_size'],
            mode='lines+markers', name=coset_name,
            hovertemplate='n=%{x}<br>last_layer=%{y}<br>coset=' + coset_name
        ))

    fig3.update_layout(
        title=dict(text=f'Last Layer Size vs n – Pancake Graph ({group_name})', y=0.95),
        xaxis_title='n', yaxis_title='Last Layer Size', yaxis_type='log',
        legend=dict(x=1.02, y=1), height=600, margin=dict(t=80)
    )
    fig3.write_html(f'{group_dir}/lastlayer.html')
    fig3.show()

    # ===== PLOT 4: Total states vs n =====
    fig4 = go.Figure()
    for coset_name in coset_names:
        c_df = df[df['coset'] == coset_name].sort_values('n')
        fig4.add_trace(go.Scatter(
            x=c_df['n'], y=c_df['total_states'],
            mode='lines+markers', name=coset_name,
            hovertemplate='n=%{x}<br>total_states=%{y}<br>coset=' + coset_name
        ))

    fig4.update_layout(
        title=dict(text=f'Total States vs n – Pancake Graph ({group_name})', y=0.95),
        xaxis_title='n', yaxis_title='Total States (= coset size)', yaxis_type='log',
        legend=dict(x=1.02, y=1), height=600, margin=dict(t=80)
    )
    fig4.write_html(f'{group_dir}/total_states.html')
    fig4.show()

    print(f"Interactive plots saved to {group_dir}/")


print("Helper functions defined!")

Helper functions defined!


---
## Group 1: "full_graph"
Full pancake graph on S_n — no coset restriction.

**Warning**: n! grows very fast. Keep max_n ≤ 12 unless you have a lot of RAM.

In [9]:
%%time
group_name = "full_graph"

# max_n=12 is recommended (12! = 479M states, ~10-20 GB RAM)
results = run_group(group_name, COSET_GROUPS[group_name],
                    min_n=3, max_n=12)
df = save_results(group_name, results, COSET_GROUPS[group_name], append=True)
plot_group_results(group_name, df)
print("\nSample data:")
display(df)

FullGraph: 100%|██████████| 10/10 [00:00<00:00, 1718.34it/s, n=12]

  FullGraph: Skipped 10 cached, computed 0 new
Completed full_graph (0 new results)


Saved: results_pancake/full_graph/data.csv (10 total rows, 0 new)


Interactive plots saved to results_pancake/full_graph/

Sample data:


,coset,n,diameter,last_layer_size,total_states,growth,central
0,FullGraph,3,3,1,6,"[1, 2, 2, 1]",None
1,FullGraph,4,4,3,24,"[1, 3, 6, 11, 3]",None
2,FullGraph,5,5,20,120,"[1, 4, 12, 35, 48, 20]",None
3,FullGraph,6,7,2,720,"[1, 5, 20, 79, 199, 281, 133, 2]",None
4,FullGraph,7,8,35,5040,"[1, 6, 30, 149, 543, 1357, 1903, 1016, 35]",None
5,FullGraph,8,9,455,40320,"[1, 7, 42, 251, 1191, 4281, 10561, 15011, 8520...",None
6,FullGraph,9,10,5804,362880,"[1, 8, 56, 391, 2278, 10666, 38015, 93585, 132...",None
7,FullGraph,10,11,73232,3628800,"[1, 9, 72, 575, 3963, 22825, 106461, 377863, 9...",None
8,FullGraph,11,13,6,39916800,"[1, 10, 90, 809, 6429, 43891, 252737, 1174766,...",None
9,FullGraph,12,14,167,479001600,"[1, 11, 110, 1099, 9883, 77937, 533397, 306478...",None


CPU times: user 243 ms, sys: 50.7 ms, total: 294 ms
Wall time: 507 ms


---
## Group 2: "different"
First D-1 elements are distinct, rest are all the same.
Coset sizes grow like n! / (n - D + 1)! — much smaller than full graph.

In [10]:
%%time
group_name = "different"

results = run_group(group_name, COSET_GROUPS[group_name],
                    min_n=5, max_n=MAX_N)
df = save_results(group_name, results, COSET_GROUPS[group_name], append=True)
plot_group_results(group_name, df)
print("\nSample data:")
display(df.head(20))

5Different: 100%|██████████| 8/8 [00:02<00:00,  2.90it/s, n=12]


  5Different: Skipped 0 cached, computed 8 new


6Different: 100%|██████████| 8/8 [00:00<00:00, 28.57it/s, n=12]


  6Different: Skipped 0 cached, computed 7 new


7Different: 100%|██████████| 8/8 [00:00<00:00, 28.05it/s, n=12] 


...

8Different: 100%|██████████| 8/8 [00:00<00:00, 17.85it/s, n=12] 

  8Different: Skipped 0 cached, computed 5 new
Completed different (26 new results)
Saved: results_pancake/different/data.csv (26 total rows, 26 new)


Interactive plots saved to results_pancake/different/

Sample data:


,coset,n,diameter,last_layer_size,total_states,growth,central
0,5Different,5,5,20,120,"[1, 4, 12, 35, 48, 20]","[0, 1, 2, 3, 4]"
1,5Different,6,6,27,360,"[1, 5, 18, 65, 127, 117, 27]","[0, 1, 2, 3, 4, 4]"
2,5Different,7,7,9,840,"[1, 6, 24, 101, 240, 314, 145, 9]","[0, 1, 2, 3, 4, 4, 4]"
3,5Different,8,7,60,1680,"[1, 7, 30, 143, 387, 634, 418, 60]","[0, 1, 2, 3, 4, 4, 4, 4]"
4,5Different,9,7,210,3024,"[1, 8, 36, 191, 568, 1100, 910, 210]","[0, 1, 2, 3, 4, 4, 4, 4, 4]"
5,5Different,10,7,540,5040,"[1, 9, 42, 245, 783, 1735, 1685, 540]","[0, 1, 2, 3, 4, 4, 4, 4, 4, 4]"
6,5Different,11,7,1155,7920,"[1, 10, 48, 305, 1032, 2562, 2807, 1155]","[0, 1, 2, 3, 4, 4, 4, 4, 4, 4, 4]"
7,5Different,12,7,2184,11880,"[1, 11, 54, 371, 1315, 3604, 4340, 2184]","[0, 1, 2, 3, 4, 4, 4, 4, 4, 4, 4, 4]"
8,6Different,6,7,2,720,"[1, 5, 20, 79, 199, 281, 133, 2]","[0, 1, 2, 3, 4, 5]"
9,6Different,7,8,1,2520,"[1, 6, 28, 131, 419, 851, 833, 250, 1]","[0, 1, 2, 3, 4, 5, 5]"


CPU times: user 1.45 s, sys: 502 ms, total: 1.95 s
Wall time: 4 s


---
## Group 3: "then"
Blocks of consecutive identical values.

In [11]:
%%time
group_name = "then"

results = run_group(group_name, COSET_GROUPS[group_name],
                    min_n=3, max_n=MAX_N)
df = save_results(group_name, results, COSET_GROUPS[group_name], append=True)
plot_group_results(group_name, df)
print("\nSample data:")
display(df.head(20))

Binary0then1: 100%|██████████| 10/10 [00:00<00:00, 1636.67it/s, n=12]


  Binary0then1: Skipped 10 cached, computed 0 new


0then1then2: 100%|██████████| 10/10 [00:00<00:00, 1843.25it/s, n=12]


  0then1then2: Skipped 10 cached, computed 0 new


0then1then2then3: 100%|██████████| 10/10 [00:00<00:00, 1714.34it/s, n=12]


...

0then1then2then3then4: 100%|██████████| 10/10 [00:00<00:00, 1931.61it/s, n=12]

  0then1then2then3then4: Skipped 8 cached, computed 0 new
Completed then (0 new results)
Saved: results_pancake/then/data.csv (37 total rows, 0 new)


Interactive plots saved to results_pancake/then/

Sample data:


,coset,n,diameter,last_layer_size,total_states,growth,central
0,0then1then2,3,3,1,6,"[1, 2, 2, 1]","[0, 1, 2]"
1,0then1then2,4,3,4,12,"[1, 3, 4, 4]","[0, 1, 2, 2]"
2,0then1then2,5,3,9,20,"[1, 4, 6, 9]","[0, 1, 2, 2, 2]"
3,0then1then2,6,5,11,90,"[1, 4, 12, 31, 31, 11]","[0, 0, 1, 1, 2, 2]"
4,0then1then2,7,6,10,210,"[1, 5, 17, 55, 75, 47, 10]","[0, 0, 1, 1, 2, 2, 2]"
5,0then1then2,8,7,6,420,"[1, 6, 22, 85, 138, 122, 40, 6]","[0, 0, 1, 1, 2, 2, 2, 2]"
6,0then1then2,9,8,7,1680,"[1, 6, 30, 135, 333, 520, 473, 175, 7]","[0, 0, 0, 1, 1, 1, 2, 2, 2]"
7,0then1then2,10,9,3,4200,"[1, 7, 38, 198, 571, 1093, 1312, 776, 201, 3]","[0, 0, 0, 1, 1, 1, 2, 2, 2, 2]"
8,0then1then2,11,10,20,9240,"[1, 8, 46, 271, 873, 1956, 2805, 2210, 935, 11...","[0, 0, 0, 1, 1, 1, 2, 2, 2, 2, 2]"
9,0then1then2,12,10,728,34650,"[1, 8, 56, 358, 1428, 4020, 7838, 9516, 7375, ...","[0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2]"


CPU times: user 145 ms, sys: 81.7 ms, total: 227 ms
Wall time: 226 ms


---
## Group 4: "coincide"
Sequential values, but last C elements are all the same.

In [12]:
%%time
group_name = "coincide"

results = run_group(group_name, COSET_GROUPS[group_name],
                    min_n=3, max_n=MAX_N)
df = save_results(group_name, results, COSET_GROUPS[group_name], append=True)
plot_group_results(group_name, df)
print("\nSample data:")
display(df.head(20))

2Coincide: 100%|██████████| 10/10 [00:21<00:00,  2.18s/it, n=12]


  2Coincide: Skipped 0 cached, computed 10 new


3Coincide: 100%|██████████| 10/10 [00:04<00:00,  2.23it/s, n=12]


  3Coincide: Skipped 0 cached, computed 9 new


4Coincide: 100%|██████████| 10/10 [00:00<00:00, 10.01it/s, n=12]


...

6Coincide: 100%|██████████| 10/10 [00:00<00:00, 46.38it/s, n=12]


  6Coincide: Skipped 0 cached, computed 6 new
Completed coincide (40 new results)
Saved: results_pancake/coincide/data.csv (40 total rows, 40 new)


Interactive plots saved to results_pancake/coincide/

Sample data:


,coset,n,diameter,last_layer_size,total_states,growth,central
0,2Coincide,3,1,2,3,"[1, 2]","[0, 1, 1]"
1,2Coincide,4,3,4,12,"[1, 3, 4, 4]","[0, 1, 2, 2]"
2,2Coincide,5,5,3,60,"[1, 4, 10, 25, 17, 3]","[0, 1, 2, 3, 3]"
3,2Coincide,6,6,27,360,"[1, 5, 18, 65, 127, 117, 27]","[0, 1, 2, 3, 4, 4]"
4,2Coincide,7,8,1,2520,"[1, 6, 28, 131, 419, 851, 833, 250, 1]","[0, 1, 2, 3, 4, 5, 5]"
5,2Coincide,8,9,27,20160,"[1, 7, 40, 229, 1003, 3200, 6487, 6790, 2376, 27]","[0, 1, 2, 3, 4, 5, 6, 6]"
6,2Coincide,9,10,496,181440,"[1, 8, 54, 365, 2014, 8728, 27607, 56795, 6120...","[0, 1, 2, 3, 4, 5, 6, 7, 7]"
7,2Coincide,10,11,7947,1814400,"[1, 9, 70, 545, 3611, 19700, 84859, 268786, 55...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 8]"
8,2Coincide,11,12,122162,19958400,"[1, 10, 88, 775, 5977, 39201, 213308, 917304, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 9]"
9,2Coincide,12,13,1878590,239500800,"[1, 11, 108, 1061, 9319, 71256, 467495, 253918...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10]"


CPU times: user 27.9 s, sys: 179 ms, total: 28.1 s
Wall time: 28.1 s


---
## Group 5: "repeats"
Repeating pattern of a short block.

In [13]:
%%time
group_name = "repeats"

results = run_group(group_name, COSET_GROUPS[group_name],
                    min_n=3, max_n=MAX_N)
df = save_results(group_name, results, COSET_GROUPS[group_name], append=True)
plot_group_results(group_name, df)
print("\nSample data:")
display(df.head(20))

Binary01Repeats: 100%|██████████| 10/10 [00:00<00:00, 1604.00it/s, n=12]


  Binary01Repeats: Skipped 10 cached, computed 0 new


Binary01Repeats_1: 100%|██████████| 10/10 [00:00<00:00, 1501.08it/s, n=12]


  Binary01Repeats_1: Skipped 10 cached, computed 0 new


012Repeats: 100%|██████████| 10/10 [00:00<00:00, 1898.82it/s, n=12]


...

011Repeats: 100%|██████████| 10/10 [00:00<00:00, 1746.54it/s, n=12]

  011Repeats: Skipped 10 cached, computed 0 new
Completed repeats (0 new results)
Saved: results_pancake/repeats/data.csv (40 total rows, 0 new)


Interactive plots saved to results_pancake/repeats/

Sample data:


,coset,n,diameter,last_layer_size,total_states,growth,central
0,011Repeats,3,1,2,3,"[1, 2]","[0, 1, 1]"
1,011Repeats,4,3,1,6,"[1, 2, 2, 1]","[0, 1, 1, 0]"
2,011Repeats,5,3,1,10,"[1, 3, 5, 1]","[0, 1, 1, 0, 1]"
3,011Repeats,6,3,3,15,"[1, 4, 7, 3]","[0, 1, 1, 0, 1, 1]"
4,011Repeats,7,4,5,35,"[1, 4, 11, 14, 5]","[0, 1, 1, 0, 1, 1, 0]"
5,011Repeats,8,5,1,56,"[1, 5, 17, 22, 10, 1]","[0, 1, 1, 0, 1, 1, 0, 1]"
6,011Repeats,9,5,4,84,"[1, 6, 21, 33, 19, 4]","[0, 1, 1, 0, 1, 1, 0, 1, 1]"
7,011Repeats,10,6,7,210,"[1, 6, 27, 63, 69, 37, 7]","[0, 1, 1, 0, 1, 1, 0, 1, 1, 0]"
8,011Repeats,11,7,1,330,"[1, 7, 36, 87, 111, 71, 16, 1]","[0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1]"
9,011Repeats,12,7,5,495,"[1, 8, 42, 114, 162, 125, 38, 5]","[0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1]"


CPU times: user 220 ms, sys: 80.5 ms, total: 300 ms
Wall time: 298 ms


---
## Summary: List all output files

In [14]:
import glob

print("Output files generated:")
for f in sorted(glob.glob(f"{OUTPUT_DIR}/**/*", recursive=True)):
    if os.path.isfile(f):
        size = os.path.getsize(f)
        print(f"  {f} ({size:,} bytes)")

Output files generated:
  results_pancake/coincide/data.csv (4,008 bytes)
  results_pancake/coincide/diameter.html (4,847,735 bytes)
  results_pancake/coincide/growth.html (4,857,288 bytes)
  results_pancake/coincide/lastlayer.html (4,847,876 bytes)
  results_pancake/coincide/total_states.html (4,847,951 bytes)
  results_pancake/data.csv (714 bytes)
  results_pancake/diameter.html (4,846,833 bytes)
  results_pancake/different/data.csv (2,871 bytes)
  results_pancake/different/diameter.html (4,847,502 bytes)
  results_pancake/different/growth.html (4,853,752 bytes)
  results_pancake/different/lastlayer.html (4,847,573 bytes)
  results_pancake/different/total_states.html (4,847,638 bytes)
  results_pancake/full_graph/data.csv (864 bytes)
  results_pancake/full_graph/diameter.html (4,846,841 bytes)
  results_pancake/full_graph/growth.html (4,849,250 bytes)
  results_pancake/full_graph/lastlayer.html (4,846,910 bytes)
  results_pancake/full_graph/total_states.html (4,846,921 bytes)
  resul

---
## Validate Against Known Pancake Diameters

Cross-check the full-graph diameters against the known sequence.

In [15]:
# Known pancake diameters (OEIS A058986)
KNOWN_DIAMETERS = {
    1: 0, 2: 1, 3: 3, 4: 4, 5: 5, 6: 7,
    7: 8, 8: 9, 9: 10, 10: 11, 11: 13, 12: 14, 13: 15
}

try:
    df_full = load_results("full_graph")
    df_fg = df_full[df_full['coset'] == 'FullGraph'].set_index('n')

    print("n | computed | known | match")
    print("-" * 35)
    all_match = True
    for n, known in sorted(KNOWN_DIAMETERS.items()):
        if n in df_fg.index:
            computed = df_fg.loc[n, 'diameter']
            match = "✓" if computed == known else "✗ MISMATCH"
            if computed != known:
                all_match = False
        else:
            computed = "—"
            match = "(not computed)"
        print(f"{n:2d} | {str(computed):8s} | {known:5d} | {match}")

    if all_match:
        print("\nAll computed diameters match known values!")
except FileNotFoundError:
    print("Run the full_graph cell first to generate data.")

n | computed | known | match
-----------------------------------
 1 | —        |     0 | (not computed)
 2 | —        |     1 | (not computed)
 3 | 3        |     3 | ✓
 4 | 4        |     4 | ✓
 5 | 5        |     5 | ✓
 6 | 7        |     7 | ✓
 7 | 8        |     8 | ✓
 8 | 9        |     9 | ✓
 9 | 10       |    10 | ✓
10 | 11       |    11 | ✓
11 | 13       |    13 | ✓
12 | 14       |    14 | ✓
13 | —        |    15 | (not computed)

All computed diameters match known values!


---
## Cross-Coset Diameter Comparison

Compare diameter trends across all coset groups side by side.

In [16]:
fig = go.Figure()

for group_name in COSET_GROUPS.keys():
    try:
        df_g = load_results(group_name)
        for coset_name in df_g['coset'].unique():
            c_df = df_g[df_g['coset'] == coset_name].sort_values('n')
            fig.add_trace(go.Scatter(
                x=c_df['n'], y=c_df['diameter'],
                mode='lines+markers',
                name=f"{group_name}/{coset_name}",
                hovertemplate='n=%{x}<br>diameter=%{y}<br>' + f"{group_name}/{coset_name}"
            ))
    except FileNotFoundError:
        pass

fig.update_layout(
    title='Diameter vs n – All Coset Groups (Pancake Graph)',
    xaxis_title='n', yaxis_title='Diameter',
    legend=dict(x=1.02, y=1),
    height=700, margin=dict(t=80)
)
fig.show()